# Revisión de Etiquetas en Archivos SAV (ENEMDU)

Este notebook extrae las etiquetas (value_labels) de los archivos .sav de ENEMDU (2021-2025)
y las compara con los mapeos documentados en `enemdu_mappings.py`.

## Objetivo
- Extraer `variable_value_labels` de cada archivo .sav por año
- Identificar variables con etiquetas
- Comparar con mapeos fallback y patrones semánticos en `enemdu_mappings.py`
- Detectar inconsistencias o nuevas etiquetas entre años

In [ ]:
#%pip install pyreadstat

In [ ]:
import pyreadstat
import pandas as pd
from pathlib import Path
import json
import sys
import re

# Agregar src al path para importar enemdu_mappings
sys.path.insert(0, '../src/processing')
from enemdu_mappings import (
    FALLBACK_AREA, FALLBACK_SEXO, FALLBACK_NIVEL_INSTRUCCION,
    FALLBACK_CONDACT, COLUMN_ALIASES,
    classify_labels, resolve_map, EDU_PATTERNS, CONDACT_PATTERNS
)

# Configurar rutas
BRONZE_DIR = Path('../data/bronze/externas/ENEMDU')
SAV_FILES = sorted(BRONZE_DIR.glob('*.sav'))

print(f"Archivos .sav encontrados: {len(SAV_FILES)}")
for f in SAV_FILES:
    print(f"  - {f.name}")

## 1. Extracción de etiquetas por año

In [ ]:
# Diccionario para almacenar etiquetas por año
etiquetas_por_ano = {}

for sav_file in SAV_FILES:
    # Extraer año del nombre: BDDenemdu_personas_YYYY_anual.sav
    ano = sav_file.name.split('_')[-2]
    
    print(f"\n{'='*60}")
    print(f"Año: {ano} | Archivo: {sav_file.name}")
    print(f"{'='*60}")
    
    # Leer archivo .sav
    df, meta = pyreadstat.read_sav(str(sav_file))
    
    # Obtener etiquetas de valores
    value_labels = meta.variable_value_labels or {}
    
    print(f"Variables con etiquetas: {len(value_labels)}")
    print(f"Dimensiones del DataFrame: {df.shape}")
    
    # Almacenar para análisis posterior
    etiquetas_por_ano[ano] = {
        'df': df,
        'meta': meta,
        'value_labels': value_labels,
        'columnas': list(df.columns)
    }
    
    # Mostrar primeras variables con etiquetas
    print(f"\nPrimeras 5 variables con etiquetas:")
    for i, (var, labels) in enumerate(list(value_labels.items())[:5]):
        print(f"  {var}: {list(labels.items())[:3]}...")

## 2. Análisis por variable: comparar etiquetas con mapeos

In [ ]:
# Crear un resumen de variables claves y sus etiquetas por año

# Variables de interés según enemdu_mappings.py
VARIABLES_INTERES = {
    'area': ['area', 'area_geografica'],  # FALLBACK_AREA
    'sexo': ['p02', 'sexo', 'p2'],         # FALLBACK_SEXO
    'nivel_instruccion': ['p10a', 'nivel_instruccion', 'nivelins', 'nnivins'],  # EDU
    'condicion_actividad': ['condact', 'condactn', 'condicion_actividad'],  # CONDACT
}

resumen_variables = {}

for ano, data in sorted(etiquetas_por_ano.items()):
    resumen_variables[ano] = {}
    value_labels = data['value_labels']
    
    for var_grupo, aliases in VARIABLES_INTERES.items():
        # Encontrar cuál alias existe en el archivo
        etiqueta_encontrada = None
        var_encontrada = None
        
        for alias in aliases:
            if alias in value_labels:
                etiqueta_encontrada = value_labels[alias]
                var_encontrada = alias
                break
        
        if etiqueta_encontrada:
            resumen_variables[ano][var_grupo] = {
                'variable': var_encontrada,
                'etiquetas': etiqueta_encontrada
            }

print("RESUMEN: Variables por año")
print(json.dumps(resumen_variables, indent=2, default=str))

## 3. Comparación: Etiquetas del SAV vs Mapeos en enemdu_mappings.py

In [ ]:
# Comparación detallada: SEXO
print("\n" + "="*70)
print("VARIABLE: SEXO (FALLBACK_SEXO)")
print("="*70)
print(f"\nMapeo esperado (fallback): {FALLBACK_SEXO}")
print("\nEtiquetas encontradas en .sav por año:")

for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n  {ano}:")
    value_labels = data['value_labels']
    
    for alias in ['p02', 'sexo', 'p2']:
        if alias in value_labels:
            labels = value_labels[alias]
            print(f"    Variable '{alias}': {dict(labels)}")
            break

In [ ]:
# Comparación detallada: NIVEL DE INSTRUCCIÓN
print("\n" + "="*70)
print("VARIABLE: NIVEL DE INSTRUCCIÓN (EDU_PATTERNS)")
print("="*70)
print(f"\nMapeo esperado (fallback): {FALLBACK_NIVEL_INSTRUCCION}")
print(f"\nPatrones de clasificación (EDU_PATTERNS):")
for cat, pattern in EDU_PATTERNS:
    print(f"  {cat}: {pattern}")

print("\n\nEtiquetas encontradas en .sav por año:")

for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n  {ano}:")
    value_labels = data['value_labels']
    
    for alias in ['p10a', 'nivel_instruccion', 'nivelins', 'nnivins']:
        if alias in value_labels:
            labels = value_labels[alias]
            print(f"    Variable '{alias}':")
            for code, text in dict(labels).items():
                print(f"      {code}: {text}")
            break

In [ ]:
# Comparación detallada: CONDICIÓN DE ACTIVIDAD
print("\n" + "="*70)
print("VARIABLE: CONDICIÓN DE ACTIVIDAD (CONDACT_PATTERNS)")
print("="*70)
print(f"\nMapeo esperado (fallback): {FALLBACK_CONDACT}")
print(f"\nPatrones de clasificación (CONDACT_PATTERNS):")
for cat, pattern in CONDACT_PATTERNS:
    print(f"  {cat}: {pattern}")

print("\n\nEtiquetas encontradas en .sav por año:")

for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n  {ano}:")
    value_labels = data['value_labels']
    
    for alias in ['condact', 'condactn', 'condicion_actividad']:
        if alias in value_labels:
            labels = value_labels[alias]
            print(f"    Variable '{alias}':")
            for code, text in dict(labels).items():
                print(f"      {code}: {text}")
            # Aplicar clasificación
            classified = classify_labels(labels, CONDACT_PATTERNS)
            print(f"    Clasificadas: {classified}")
            break

In [ ]:
# Comparación detallada: ÁREA GEOGRÁFICA
print("\n" + "="*70)
print("VARIABLE: ÁREA GEOGRÁFICA (FALLBACK_AREA)")
print("="*70)
print(f"\nMapeo esperado (fallback): {FALLBACK_AREA}")
print("\nEtiquetas encontradas en .sav por año:")

for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n  {ano}:")
    value_labels = data['value_labels']
    
    for alias in ['area', 'area_geografica']:
        if alias in value_labels:
            labels = value_labels[alias]
            print(f"    Variable '{alias}': {dict(labels)}")
            break

## 4. Listado completo de variables con etiquetas por año

In [ ]:
# Tabla resumen: todas las variables con etiquetas
for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n{'='*70}")
    print(f"AÑO {ano} - Variables con etiquetas ({len(data['value_labels'])})")
    print(f"{'='*70}")
    
    value_labels = data['value_labels']
    for var_name in sorted(value_labels.keys()):
        labels = value_labels[var_name]
        n_labels = len(labels)
        print(f"\n  {var_name} ({n_labels} etiquetas):")
        for code, text in dict(labels).items():
            print(f"    {code}: {text}")

## 5. Análisis de diferencias entre años

In [ ]:
# Identificar variables que existen en algunos años pero no en otros
all_vars = set()
for ano, data in etiquetas_por_ano.items():
    all_vars.update(data['value_labels'].keys())

print("\nVariables que cambian entre años:")
print("="*70)

for var in sorted(all_vars):
    presencia = {}
    for ano in sorted(etiquetas_por_ano.keys()):
        presencia[ano] = var in etiquetas_por_ano[ano]['value_labels']
    
    # Mostrar solo si cambia entre años
    if not all(presencia.values()) or not all(v == presencia['2021'] for v in presencia.values()):
        print(f"\n{var}:")
        for ano, existe in sorted(presencia.items()):
            estado = "✓ Sí" if existe else "✗ No"
            print(f"  {ano}: {estado}")

## 6. Prueba de funciones classify_labels y resolve_map

In [ ]:
# Probar resolve_map para diferentes variables y años
print("\nPrueba de resolve_map():")
print("="*70)

test_cases = [
    ('nivel_instruccion', ['p10a', 'nivel_instruccion', 'nivelins', 'nnivins'], EDU_PATTERNS, FALLBACK_NIVEL_INSTRUCCION),
    ('condicion_actividad', ['condact', 'condactn', 'condicion_actividad'], CONDACT_PATTERNS, FALLBACK_CONDACT),
]

for var_name, aliases, patterns, fallback in test_cases:
    print(f"\n\n{var_name.upper()}")
    print("-" * 70)
    
    for ano, data in sorted(etiquetas_por_ano.items()):
        value_labels = data['value_labels']
        
        # Encontrar alias que existe
        labels_dict = None
        alias_usado = None
        for alias in aliases:
            if alias in value_labels:
                labels_dict = value_labels[alias]
                alias_usado = alias
                break
        
        if labels_dict:
            result_map, procedencia = resolve_map(labels_dict, patterns, fallback)
            print(f"\n  {ano} (variable '{alias_usado}'):")
            print(f"    Procedencia: {procedencia}")
            print(f"    Mapeo: {result_map}")